# V-JEPA 2 vs Qwen2.5-VL — HMDB51 Benchmark (Kaggle)

**Dataset**: HMDB51 — 6,754 videos, 51 action classes
**Models**: V-JEPA 2 (654M) vs Qwen2.5-VL (7B)
**Method**: Frozen features + classifiers (no fine-tuning)

**Kaggle Setup**:
- Accelerator: **GPU P100** or **GPU T4 x2**
- Internet: **ON**
- Features saved to `/kaggle/working/` (persists as output)
- To resume: re-run all cells — skips already-extracted features

In [2]:
!pip install -q transformers accelerate bitsandbytes peft datasets huggingface_hub opencv-python-headless scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 99.4 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is inco

In [3]:
import torch, sys, os, time, json, gc, random, tempfile
import numpy as np
import torch.nn as nn
from pathlib import Path
from collections import Counter
from PIL import Image
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, classification_report,
    confusion_matrix, top_k_accuracy_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
import cv2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    gpu = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f'GPU: {gpu} ({mem/1e9:.1f} GB)')
else:
    print('WARNING: No GPU! Enable in Settings -> Accelerator')

SAVE_DIR = '/kaggle/working'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Output: {SAVE_DIR}')

GPU: Tesla T4 (15.6 GB)
Output: /kaggle/working


## Step 1: Load HMDB51 Dataset

In [4]:
SEED = 42
NUM_FRAMES = 8
IMG_SIZE = 224
TRAIN_SPLIT = 0.8

HMDB51_CLASSES = [
    'brush_hair','cartwheel','catch','chew','clap','climb','climb_stairs',
    'dive','draw_sword','dribble','drink','eat','fall_floor','fencing',
    'flic_flac','golf','handstand','hit','hug','jump','kick','kick_ball',
    'kiss','laugh','pick','pour','pullup','punch','push','pushup',
    'ride_bike','ride_horse','run','shake_hands','shoot_ball','shoot_bow',
    'shoot_gun','sit','situp','smile','smoke','somersault','stand',
    'swing_baseball','sword','sword_exercise','talk','throw','turn','walk','wave',
]
NUM_CLASSES = len(HMDB51_CLASSES)
print(f'{NUM_CLASSES} action classes')

def load_frames_from_bytes(video_bytes, n_frames=8, size=224):
    """Extract frames from video bytes."""
    with tempfile.NamedTemporaryFile(suffix='.avi', delete=False) as f:
        f.write(video_bytes)
        tmp = f.name
    try:
        cap = cv2.VideoCapture(tmp)
        if not cap.isOpened(): return None
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total < 1: cap.release(); return None
        idxs = np.linspace(0, total - 1, n_frames, dtype=int)
        frames = []
        for i in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ok, fr = cap.read()
            if not ok: fr = np.zeros((size, size, 3), dtype=np.uint8)
            fr = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
            fr = cv2.resize(fr, (size, size))
            frames.append(Image.fromarray(fr))
        cap.release()
        return frames
    finally:
        os.unlink(tmp)

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

51 action classes


In [5]:
import os
import zipfile
from huggingface_hub import hf_hub_download
from collections import Counter

# 1. Define clean working directories on Kaggle
download_dir = "/kaggle/working/hmdb51_raw"
extract_dir = "/kaggle/working/hmdb51_dataset"
os.makedirs(download_dir, exist_ok=True)
os.makedirs(extract_dir, exist_ok=True)

print("Downloading the raw archive directly from the Hub to bypass schema validation...")
try:
    # Safely grab the raw zip file using the Hub file client
    archive_path = hf_hub_download(
        repo_id="jili5044/hmdb51",
        filename="hmdb51.zip", # Targets the raw underlying asset file
        repo_type="dataset",
        local_dir=download_dir
    )
    print(f"Download complete! File saved at: {archive_path}")
    
    # 2. Extracting raw assets
    print("Extracting dataset assets...")
    with zipfile.ZipFile(archive_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print(f"Extraction complete! Files are ready at: {extract_dir}")
    
    # 3. Read and inspect extracted folders/classes directly from storage
    # This completely avoids any DatasetGenerationErrors
    extracted_items = os.listdir(extract_dir)
    print(f"\nExtracted contents snippet: {extracted_items[:5]}")
    
    # Let's map out what action classes are present in your workspace
    video_classes = []
    for root, dirs, files in os.walk(extract_dir):
        for file in files:
            if file.endswith(('.mp4', '.avi', '.mkv')):
                # Infer action category name from parent directory name
                classname = os.path.basename(root)
                video_classes.append(classname)
                
    counts = Counter(video_classes)
    print(f"\nSuccessfully Verified Local Classes ({len(counts)} structural categories found):")
    for name in sorted(counts.keys()):
        print(f"  {name:<20} Samples: {counts[name]:>4}")

except Exception as e:
    print(f"An processing issue occurred: {e}")

hmdb51.zip:   0%|          | 0.00/2.12G [00:00<?, ?B/s]

Download complete! File saved at: /kaggle/working/hmdb51_raw/hmdb51.zip
Extracting dataset assets...
Extraction complete! Files are ready at: /kaggle/working/hmdb51_dataset

Extracted contents snippet: ['hmdb51']

Successfully Verified Local Classes (51 structural categories found):
  brush_hair           Samples:  103
  cartwheel            Samples:  103
  catch                Samples:  102
  chew                 Samples:  109
  clap                 Samples:  130
  climb                Samples:  108
  climb_stairs         Samples:  112
  dive                 Samples:  127
  draw_sword           Samples:  103
  dribble              Samples:  145
  drink                Samples:  164
  eat                  Samples:  108
  fall_floor           Samples:  136
  fencing              Samples:  116
  flic_flac            Samples:  107
  golf                 Samples:  104
  handstand            Samples:  113
  hit                  Samples:  127
  hug                  Samples:  118
  jump       

In [6]:
import os

# 1. Point to the local folder where the previous script unzipped everything
extract_dir = "/kaggle/working/hmdb51_dataset"

# 2. Reconstruct a flat list of samples (video path and its label) directly from disk
ds = []
for root, dirs, files in os.walk(extract_dir):
    for file in files:
        if file.endswith(('.mp4', '.avi', '.mkv')):
            video_path = os.path.join(root, file)
            # Label is simply the name of the folder the video is in
            label_name = os.path.basename(root) 
            
            ds.append({
                'video_path': video_path,
                'label': label_name
            })

print(f"Bypassed NameError! Created a local 'ds' list containing {len(ds)} video paths.")
print(f"Sample structural schema: {ds[0] if len(ds) > 0 else 'No videos found'}")

Bypassed NameError! Created a local 'ds' list containing 6754 video paths.
Sample structural schema: {'video_path': '/kaggle/working/hmdb51_dataset/hmdb51/handstand/Outdoor-Turnen_handstand_f_cm_np1_fr_med_0.avi', 'label': 'handstand'}


## Step 2: Extract V-JEPA 2 Features
Frozen backbone → mean-pooled last hidden state (1280-dim per video). Checkpoint saved after extraction.

In [7]:
from transformers import VJEPA2Model, VJEPA2VideoProcessor
import os
import torch
import time
import gc

VJEPA_FEAT_PATH = os.path.join(SAVE_DIR, 'vjepa2_hmdb51_features.pt')
VJEPA_CKPT_PATH = os.path.join(SAVE_DIR, 'vjepa2_hmdb51_ckpt.pt')  # Incremental checkpoint
CKPT_EVERY = 500  # Save progress every 500 videos

if os.path.exists(VJEPA_FEAT_PATH):
    print(f'V-JEPA features COMPLETE: {VJEPA_FEAT_PATH}')
    vjepa_data = torch.load(VJEPA_FEAT_PATH, map_location='cpu', weights_only=True)
    print(f'  Shape: {vjepa_data["features"].shape}')
else:
    print('Loading V-JEPA 2 model...')
    vjepa_proc = VJEPA2VideoProcessor.from_pretrained('facebook/vjepa2-vith-fpc64-256')
    vjepa_model = VJEPA2Model.from_pretrained(
        'facebook/vjepa2-vith-fpc64-256',
        torch_dtype=torch.float16, attn_implementation='sdpa',
    ).to(device).eval()
    print(f'  Loaded on {device}')

    # Resume from checkpoint if exists
    features, labels, start_idx = [], [], 0
    if os.path.exists(VJEPA_CKPT_PATH):
        ckpt = torch.load(VJEPA_CKPT_PATH, map_location='cpu', weights_only=True)
        features = list(ckpt['features'].unbind(0))
        labels = ckpt['labels'].tolist()
        start_idx = int(ckpt['next_idx'])
        print(f'  >> Resuming from checkpoint: {len(features)} videos done, starting at idx {start_idx}')
        del ckpt; gc.collect()
    else:
        print(f'  Starting fresh (0/{len(ds)})')

    errors = 0
    t0 = time.time()
    total = len(ds)

    for i in range(start_idx, total):
        try:
            sample = ds[i]
            
            # 1. Extract the file path string from our manual dictionary schema
            video_path = sample['video_path']
            label_name = sample['label']
            
            # 2. Convert the string category name to its required class integer index
            if label_name in HMDB51_CLASSES:
                label_id = HMDB51_CLASSES.index(label_name)
            else:
                # Fallback if there's a naming mismatch with your predefined listing
                label_id = int(label_name) if label_name.isdigit() else 0

            # 3. Read the raw file directly from local storage paths
            with open(video_path, 'rb') as f:
                vbytes = f.read()

            if vbytes is None: 
                errors += 1
                continue

            frames = load_frames_from_bytes(vbytes, NUM_FRAMES, IMG_SIZE)
            if frames is None: 
                errors += 1
                continue

            inputs = vjepa_proc(frames, return_tensors='pt')
            inputs = {k: v.to(device) for k, v in inputs.items()}

            with torch.no_grad(), torch.amp.autocast('cuda'):
                out = vjepa_model(**inputs)
                feat = out.last_hidden_state.mean(dim=1).cpu().float().squeeze(0)

            features.append(feat)
            labels.append(label_id)  # Appending the numerical class ID index

            # Progress Tracking
            if (i + 1) % 100 == 0:
                elapsed = time.time() - t0
                done_this_run = i - start_idx + 1
                speed = done_this_run / elapsed
                eta = (total - i - 1) / speed / 60
                print(f'  [{i+1}/{total}] {len(features)} extracted | {speed:.1f} vid/s | ETA: {eta:.0f}min')

            # ===== CHECKPOINT every 500 videos =====
            if (i + 1) % CKPT_EVERY == 0:
                torch.save({
                    'features': torch.stack(features),
                    'labels': torch.tensor(labels, dtype=torch.long),
                    'next_idx': i + 1,
                }, VJEPA_CKPT_PATH)
                print(f'    >> Checkpoint saved ({len(features)} videos, next={i+1})')

            if (i + 1) % 500 == 0:
                torch.cuda.empty_cache()

        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                torch.cuda.empty_cache(); errors += 1; continue
            raise
        except Exception as e:
            errors += 1
            if errors <= 10: 
                print(f'  Error at index {i}: {type(e).__name__}: {e}')
            continue

    # Save final features
    if len(features) > 0:
        features_t = torch.stack(features)
        labels_t = torch.tensor(labels, dtype=torch.long)
        torch.save({'features': features_t, 'labels': labels_t}, VJEPA_FEAT_PATH)
        
        # Remove checkpoint (no longer needed)
        if os.path.exists(VJEPA_CKPT_PATH): 
            os.remove(VJEPA_CKPT_PATH)
            
        elapsed = time.time() - t0
        print(f'\nV-JEPA DONE: {features_t.shape[0]} videos | {features_t.shape[1]}-dim | {elapsed/60:.1f}min | {errors} errors')
    else:
        print("\nPipeline failed: No features were successfully generated. Check video loading logs.")

    del vjepa_model, vjepa_proc
    torch.cuda.empty_cache(); gc.collect()

## Step 3: Extract Qwen2.5-VL Features
4-bit quantized → hidden states (3584-dim per video). Checkpoint saved after extraction.

In [8]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
import os
import torch
import time
import gc

QWEN_FEAT_PATH = os.path.join(SAVE_DIR, 'qwen25vl_hmdb51_features.pt')
QWEN_CKPT_PATH = os.path.join(SAVE_DIR, 'qwen25vl_hmdb51_ckpt.pt')  # Incremental checkpoint
CKPT_EVERY = 500  # Save progress every 500 videos

if os.path.exists(QWEN_FEAT_PATH):
    print(f'Qwen features COMPLETE: {QWEN_FEAT_PATH}')
    qwen_data = torch.load(QWEN_FEAT_PATH, map_location='cpu', weights_only=True)
    print(f'  Shape: {qwen_data["features"].shape}')
else:
    print('Loading Qwen2.5-VL (4-bit)...')
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
    )
    qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        'Qwen/Qwen2.5-VL-7B-Instruct',
        quantization_config=bnb, device_map='auto', torch_dtype=torch.float16,
    )
    qwen_proc = AutoProcessor.from_pretrained('Qwen/Qwen2.5-VL-7B-Instruct')
    qwen_model.eval()
    qwen_dev = next(qwen_model.parameters()).device
    print(f'  Loaded (4-bit) on {qwen_dev}')

    # Resume from checkpoint if exists
    features, labels, start_idx = [], [], 0
    if os.path.exists(QWEN_CKPT_PATH):
        ckpt = torch.load(QWEN_CKPT_PATH, map_location='cpu', weights_only=True)
        features = list(ckpt['features'].unbind(0))
        labels = ckpt['labels'].tolist()
        start_idx = int(ckpt['next_idx'])
        print(f'  >> Resuming from checkpoint: {len(features)} videos done, starting at idx {start_idx}')
        del ckpt; gc.collect()
    else:
        print(f'  Starting fresh (0/{len(ds)})')

    errors = 0
    t0 = time.time()
    total = len(ds)

    for i in range(start_idx, total):
        try:
            sample = ds[i]
            
            # 1. FIXED: Point directly to the storage file path
            video_path = sample['video_path']
            label_name = sample['label']
            
            # 2. FIXED: Map textual labels to explicit integer indices
            if label_name in HMDB51_CLASSES:
                label_id = HMDB51_CLASSES.index(label_name)
            else:
                label_id = int(label_name) if label_name.isdigit() else 0

            # 3. FIXED: Load files straight from local directory
            with open(video_path, 'rb') as f: 
                vbytes = f.read()

            if vbytes is None: 
                errors += 1; continue

            frames = load_frames_from_bytes(vbytes, 4, IMG_SIZE)  # 4 frames for Qwen
            if frames is None: 
                errors += 1; continue

            messages = [{'role': 'user', 'content': [
                {'type': 'video', 'video': frames},
                {'type': 'text', 'text': 'Describe.'},
            ]}]
            text = qwen_proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = qwen_proc(text=[text], videos=[frames], return_tensors='pt', padding=True)
            inputs = {k: v.to(qwen_dev) if hasattr(v, 'to') else v for k, v in inputs.items()}

            with torch.no_grad():
                out = qwen_model(**inputs, output_hidden_states=True)
                feat = out.hidden_states[-1].mean(dim=1).cpu().float().squeeze(0)

            features.append(feat)
            labels.append(label_id)  # Appending integer class assignment

            # Progress monitoring
            if (i + 1) % 50 == 0:
                elapsed = time.time() - t0
                done_this_run = i - start_idx + 1
                speed = done_this_run / elapsed
                eta = (total - i - 1) / speed / 60
                print(f'  [{i+1}/{total}] {len(features)} extracted | {speed:.1f} vid/s | ETA: {eta:.0f}min')

            # ===== CHECKPOINT every 500 videos =====
            if (i + 1) % CKPT_EVERY == 0:
                torch.save({
                    'features': torch.stack(features),
                    'labels': torch.tensor(labels, dtype=torch.long),
                    'next_idx': i + 1,
                }, QWEN_CKPT_PATH)
                print(f'    >> Checkpoint saved ({len(features)} videos, next={i+1})')

            if (i + 1) % 200 == 0:
                torch.cuda.empty_cache()

        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                torch.cuda.empty_cache(); errors += 1; continue
            raise
        except Exception as e:
            errors += 1
            if errors <= 5: 
                print(f'  Error at index {i}: {type(e).__name__}: {e}')
            continue

    # Save final features
    if len(features) > 0:
        features_t = torch.stack(features)
        labels_t = torch.tensor(labels, dtype=torch.long)
        torch.save({'features': features_t, 'labels': labels_t}, QWEN_FEAT_PATH)

        # Remove checkpoint (no longer needed)
        if os.path.exists(QWEN_CKPT_PATH): os.remove(QWEN_CKPT_PATH)

        elapsed = time.time() - t0
        print(f'\nQwen DONE: {features_t.shape[0]} videos | {features_t.shape[1]}-dim | {elapsed/60:.1f}min | {errors} errors')
    else:
        print("\nPipeline failed: Feature stack empty. Double check local file availability.")

    del qwen_model, qwen_proc
    torch.cuda.empty_cache(); gc.collect()

Loading Qwen2.5-VL (4-bit)...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

... (4 outputs left out)

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  Loaded (4-bit) on cuda:0
  Starting fresh (0/6754)
  [50/6754] 50 extracted | 2.1 vid/s | ETA: 54min
  [100/6754] 100 extracted | 2.1 vid/s | ETA: 52min
  [150/6754] 150 extracted | 2.1 vid/s | ETA: 52min
  [200/6754] 200 extracted | 2.1 vid/s | ETA: 52min
  [250/6754] 250 extracted | 2.1 vid/s | ETA: 52min
  [300/6754] 300 extracted | 2.1 vid/s | ETA: 52min
  [350/6754] 350 extracted | 2.0 vid/s | ETA: 52min
  [400/6754] 400 extracted | 2.0 vid/s | ETA: 52min
  [450/6754] 450 extracted | 2.0 vid/s | ETA: 52min
  [500/6754] 500 extracted | 2.0 vid/s | ETA: 52min
    >> Checkpoint saved (500 videos, next=500)
  [550/6754] 550 extracted | 2.0 vid/s | ETA: 52min
  [600/6754] 600 extracted | 2.0 vid/s | ETA: 51min
  [650/6754] 650 extracted | 2.0 vid/s | ETA: 51min
  [700/6754] 700 extracted | 2.0 vid/s | ETA: 50min
  [750/6754] 750 extracted | 2.0 vid/s | ETA: 50min
  [800/6754] 800 extracted | 2.0 vid/s | ETA: 50min
  [850/6754] 850 extracted | 2.0 vid/s | ETA: 49min
  [900/6754] 900 e

## Step 4: Run Classifiers & Compare
No GPU needed from here. Multiple classifiers tested on frozen features.

In [9]:
# Load features
vjepa_data = torch.load(VJEPA_FEAT_PATH, map_location='cpu', weights_only=True)
qwen_data = torch.load(QWEN_FEAT_PATH, map_location='cpu', weights_only=True)

vjepa_feat = vjepa_data['features'].numpy()
qwen_feat = qwen_data['features'].numpy()
vjepa_labels = vjepa_data['labels'].numpy()
qwen_labels = qwen_data['labels'].numpy()

# Use common samples
n = min(len(vjepa_labels), len(qwen_labels))
vjepa_feat, qwen_feat = vjepa_feat[:n], qwen_feat[:n]
labels = vjepa_labels[:n]

print(f'V-JEPA features: {vjepa_feat.shape}')
print(f'Qwen features:   {qwen_feat.shape}')
print(f'Videos: {n} | Classes: {NUM_CLASSES}')

# Train/eval split
n_tr = int(n * TRAIN_SPLIT)
perm = np.random.RandomState(SEED).permutation(n)
tr_idx, ev_idx = perm[:n_tr], perm[n_tr:]
y_tr, y_ev = labels[tr_idx], labels[ev_idx]
print(f'Train: {len(tr_idx)} | Eval: {len(ev_idx)}')

V-JEPA features: (6754, 1280)
Qwen features:   (6754, 3584)
Videos: 6754 | Classes: 51
Train: 5403 | Eval: 1351


## Step 4b: PyTorch Neural Network Classifiers (Enhanced)
Custom neural networks for feature classification: **DeepMLP**, **ResMLP** (residual connections), **1D-CNN**, **Mini Transformer** encoder, **AttentionPoolMLP** (learned attention), and **CosineClassifier** (magnitude-invariant). All trained on PCA-reduced features with:
- **Label smoothing** (0.1) for regularization
- **Feature-space mixup** augmentation for better generalization
- Early stopping with cosine LR scheduling

In [ ]:
# ──────────────────────────────────────────────────────────────
#  PyTorch Neural Network Architectures for Feature Classification
#  Enhanced with AttentionPool, GeM pooling, and label smoothing
# ──────────────────────────────────────────────────────────────
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

LABEL_SMOOTHING = 0.1  # Regularization: prevents overconfident predictions

# ── 1. DeepMLP: 4-layer MLP with BatchNorm, Dropout, GELU ──
class DeepMLP(nn.Module):
    def __init__(self, in_dim, num_classes, hidden_dims=(512, 256, 128, 64), dropout=0.3):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        self.backbone = nn.Sequential(*layers)
        self.head = nn.Linear(prev, num_classes)

    def forward(self, x):
        return self.head(self.backbone(x))


# ── 2. ResMLP: MLP with residual / skip connections ──
class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim),
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(x + self.net(x))

class ResMLP(nn.Module):
    def __init__(self, in_dim, num_classes, hidden=256, n_blocks=3, dropout=0.2):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.GELU())
        self.blocks = nn.Sequential(*[ResBlock(hidden, dropout) for _ in range(n_blocks)])
        self.head = nn.Linear(hidden, num_classes)

    def forward(self, x):
        return self.head(self.blocks(self.proj(x)))


# ── 3. MiniTransformer: Transformer encoder on chunked features ──
class MiniTransformer(nn.Module):
    def __init__(self, in_dim, num_classes, d_model=64, nhead=4, num_layers=2, dropout=0.2):
        super().__init__()
        self.chunk_size = d_model
        # Pad feature dim to be divisible by d_model
        self.n_chunks = (in_dim + d_model - 1) // d_model
        self.pad_dim = self.n_chunks * d_model
        self.proj = nn.Linear(d_model, d_model)  # per-chunk linear
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, self.n_chunks + 1, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=dropout, activation='gelu', batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x):
        B = x.size(0)
        # Pad and reshape into chunks: (B, n_chunks, d_model)
        if x.size(1) < self.pad_dim:
            x = F.pad(x, (0, self.pad_dim - x.size(1)))
        x = x.view(B, self.n_chunks, self.chunk_size)
        x = self.proj(x)
        # Prepend CLS token
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)  # (B, n_chunks+1, d_model)
        x = x + self.pos_embed
        x = self.encoder(x)
        x = self.norm(x[:, 0])  # CLS token output
        return self.head(x)


# ── 4. AttentionPoolMLP: MLP with learned attention pooling ──
# Learns which feature dimensions are most discriminative
class AttentionPoolMLP(nn.Module):
    def __init__(self, in_dim, num_classes, hidden=256, dropout=0.3):
        super().__init__()
        # Split features into chunks and apply attention across them
        self.chunk_size = 64
        self.n_chunks = (in_dim + self.chunk_size - 1) // self.chunk_size
        self.pad_dim = self.n_chunks * self.chunk_size
        # Attention over chunks
        self.attn_proj = nn.Linear(self.chunk_size, 1)
        self.backbone = nn.Sequential(
            nn.Linear(self.chunk_size, hidden), nn.BatchNorm1d(hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.GELU(), nn.Dropout(dropout),
        )
        self.head = nn.Linear(hidden, num_classes)

    def forward(self, x):
        B = x.size(0)
        if x.size(1) < self.pad_dim:
            x = F.pad(x, (0, self.pad_dim - x.size(1)))
        x = x.view(B, self.n_chunks, self.chunk_size)  # (B, C, chunk)
        # Attention-weighted pooling across chunks
        attn_w = torch.softmax(self.attn_proj(x), dim=1)  # (B, C, 1)
        x = (x * attn_w).sum(dim=1)  # (B, chunk_size)
        return self.head(self.backbone(x))


# ── 5. CosineClassifier: Cosine-similarity based head ──
# Removes magnitude bias — only direction matters (great for normalized features)
class CosineClassifier(nn.Module):
    def __init__(self, in_dim, num_classes, hidden=256, scale=16.0, dropout=0.3):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.GELU(), nn.Dropout(dropout),
        )
        self.weight = nn.Parameter(torch.randn(num_classes, hidden))
        self.scale = nn.Parameter(torch.tensor(scale))

    def forward(self, x):
        x = self.backbone(x)
        # Cosine similarity: normalize both feature and weight vectors
        x_norm = F.normalize(x, p=2, dim=-1)
        w_norm = F.normalize(self.weight, p=2, dim=-1)
        return self.scale * x_norm @ w_norm.T


# ── Training harness with label smoothing + mixup ──
def mixup_data(x, y, alpha=0.2):
    """Feature-space mixup: interpolate training samples for better generalization."""
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1 - lam)  # Ensure lam >= 0.5
    idx = torch.randperm(x.size(0))
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def train_pytorch_model(model, X_tr, y_tr, X_ev, y_ev,
                        epochs=200, lr=1e-3, batch_size=64, patience=20,
                        use_mixup=True, verbose=False):
    """Train a PyTorch classifier with early stopping, label smoothing, and mixup."""
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    # Convert to tensors
    Xt = torch.tensor(X_tr, dtype=torch.float32)
    yt = torch.tensor(y_tr, dtype=torch.long)
    Xe = torch.tensor(X_ev, dtype=torch.float32).to(device)
    ye = torch.tensor(y_ev, dtype=torch.long).to(device)

    ds = TensorDataset(Xt, yt)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=False)

    best_acc, best_state, wait = 0, None, 0
    for epoch in range(epochs):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            if use_mixup and xb.size(0) > 1:
                xb_mix, ya, yb_mix, lam = mixup_data(xb, yb, alpha=0.2)
                logits = model(xb_mix)
                loss = lam * F.cross_entropy(logits, ya, label_smoothing=LABEL_SMOOTHING) + \
                       (1 - lam) * F.cross_entropy(logits, yb_mix, label_smoothing=LABEL_SMOOTHING)
            else:
                loss = F.cross_entropy(model(xb), yb, label_smoothing=LABEL_SMOOTHING)
            opt.zero_grad(); loss.backward(); opt.step()
        scheduler.step()

        # Eval
        model.eval()
        with torch.no_grad():
            logits = model(Xe)
            acc = (logits.argmax(1) == ye).float().mean().item()
        if acc > best_acc:
            best_acc = acc; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}; wait = 0
        else:
            wait += 1
            if wait >= patience:
                if verbose: print(f'    Early stop at epoch {epoch+1}')
                break

    # Load best and predict
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    model = model.to(device)
    with torch.no_grad():
        logits = model(Xe)
        probs = F.softmax(logits, dim=1).cpu().numpy()
        preds = logits.argmax(1).cpu().numpy()
    return preds, probs

print('PyTorch neural network architectures defined \u2713')
print('  \u2022 DeepMLP         \u2014 4-layer MLP with BatchNorm + GELU + Dropout')
print('  \u2022 ResMLP          \u2014 MLP with residual skip connections (3 blocks)')
print('  \u2022 MiniTransformer \u2014 Transformer encoder on chunked features')
print('  \u2022 AttentionPoolMLP\u2014 MLP with learned attention pooling')
print('  \u2022 CosineClassifier\u2014 Cosine-similarity head (magnitude-invariant)')
print(f'  \u2022 Label smoothing = {LABEL_SMOOTHING}')
print(f'  \u2022 Feature-space mixup augmentation enabled')

In [10]:
def run_classifiers(X_tr, y_tr, X_ev, y_ev, name):
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_ev_s = scaler.transform(X_ev)
    pca = PCA(n_components=0.95, random_state=SEED)
    X_tr_p = pca.fit_transform(X_tr_s)
    X_ev_p = pca.transform(X_ev_s)
    in_dim = X_tr_p.shape[1]
    print(f'  PCA: {X_tr_s.shape[1]} -> {in_dim} dims')

    # ── Sklearn classifiers ──
    classifiers = {
        'SVM-RBF':      SVC(kernel='rbf', C=10, gamma='scale', class_weight='balanced',
                            random_state=SEED, probability=True),
        'KNN-7':        KNeighborsClassifier(n_neighbors=7, n_jobs=-1),
        'RandomForest': RandomForestClassifier(n_estimators=500, class_weight='balanced',
                                               random_state=SEED, n_jobs=-1),
        'MLP':          MLPClassifier(hidden_layer_sizes=(512, 256), max_iter=1000, alpha=1e-4,
                                      random_state=SEED, early_stopping=True, learning_rate='adaptive'),
        'LogReg':       LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000,
                                           random_state=SEED, multi_class='multinomial'),
    }

    print(f'\n  {"Clf":<20} {"Acc":>7} {"Top-5":>7} {"F1":>7} {"MCC":>7}  {"Type"}')
    print(f'  {"-"*62}')

    best_name, best_f1, best_preds, best_prob = '', 0, None, None

    # Run sklearn classifiers
    for cname, clf in classifiers.items():
        print(f'  {cname}...', end=' ', flush=True)
        clf.fit(X_tr_p, y_tr)
        preds = clf.predict(X_ev_p)
        prob = clf.predict_proba(X_ev_p) if hasattr(clf, 'predict_proba') else None
        acc = accuracy_score(y_ev, preds)
        f1 = f1_score(y_ev, preds, average='macro', zero_division=0)
        mcc = matthews_corrcoef(y_ev, preds)
        t5 = ''
        if prob is not None:
            try: t5 = f'{top_k_accuracy_score(y_ev, prob, k=5, labels=range(NUM_CLASSES))*100:6.1f}%'
            except: t5 = '  N/A  '
        mark = ''
        if f1 > best_f1:
            best_f1 = f1; best_name = cname; best_preds = preds; best_prob = prob; mark = ' *'
        print(f'\r  {cname:<20} {acc*100:6.1f}% {t5:>7} {f1*100:6.1f}% {mcc:6.3f}{mark}  sklearn')

    # ── PyTorch neural network classifiers ──
    torch.manual_seed(SEED)
    nn_models = {
        'DeepMLP':        DeepMLP(in_dim, NUM_CLASSES, hidden_dims=(512, 256, 128, 64), dropout=0.3),
        'ResMLP':         ResMLP(in_dim, NUM_CLASSES, hidden=256, n_blocks=3, dropout=0.2),
        'Transformer':    MiniTransformer(in_dim, NUM_CLASSES, d_model=64, nhead=4, num_layers=2, dropout=0.2),
        'AttnPoolMLP':    AttentionPoolMLP(in_dim, NUM_CLASSES, hidden=256, dropout=0.3),
        'CosineClf':      CosineClassifier(in_dim, NUM_CLASSES, hidden=256, scale=16.0, dropout=0.3),
    }

    for cname, model in nn_models.items():
        n_params = sum(p.numel() for p in model.parameters())
        print(f'  Training {cname} ({n_params:,} params)...', end=' ', flush=True)
        torch.manual_seed(SEED)
        preds, prob = train_pytorch_model(
            model, X_tr_p, y_tr, X_ev_p, y_ev,
            epochs=300, lr=1e-3, batch_size=64, patience=30,
        )
        acc = accuracy_score(y_ev, preds)
        f1 = f1_score(y_ev, preds, average='macro', zero_division=0)
        mcc = matthews_corrcoef(y_ev, preds)
        t5 = ''
        if prob is not None:
            try: t5 = f'{top_k_accuracy_score(y_ev, prob, k=5, labels=range(NUM_CLASSES))*100:6.1f}%'
            except: t5 = '  N/A  '
        mark = ''
        if f1 > best_f1:
            best_f1 = f1; best_name = cname; best_preds = preds; best_prob = prob; mark = ' *'
        print(f'\r  {cname:<20} {acc*100:6.1f}% {t5:>7} {f1*100:6.1f}% {mcc:6.3f}{mark}  pytorch')

    # ── Compute full metrics for best ──
    m = dict(
        accuracy=accuracy_score(y_ev, best_preds),
        precision_macro=precision_score(y_ev, best_preds, average='macro', zero_division=0),
        precision_weighted=precision_score(y_ev, best_preds, average='weighted', zero_division=0),
        recall_macro=recall_score(y_ev, best_preds, average='macro', zero_division=0),
        recall_weighted=recall_score(y_ev, best_preds, average='weighted', zero_division=0),
        f1_macro=f1_score(y_ev, best_preds, average='macro', zero_division=0),
        f1_weighted=f1_score(y_ev, best_preds, average='weighted', zero_division=0),
        mcc=matthews_corrcoef(y_ev, best_preds),
        cohen_kappa=cohen_kappa_score(y_ev, best_preds),
    )
    if best_prob is not None:
        try: m['top5_accuracy'] = top_k_accuracy_score(y_ev, best_prob, k=5, labels=range(NUM_CLASSES))
        except: pass
    report = classification_report(y_ev, best_preds, target_names=HMDB51_CLASSES, zero_division=0)
    return best_name, m, report

# ===== V-JEPA 2 =====
print('=' * 60)
print('  V-JEPA 2 (654M) \u2014 Self-supervised video encoder')
print('=' * 60)
vj_name, vj_m, vj_report = run_classifiers(vjepa_feat[tr_idx], y_tr, vjepa_feat[ev_idx], y_ev, 'V-JEPA')
print(f'\n  Best: {vj_name}')
print(vj_report)

# ===== Qwen2.5-VL =====
print('=' * 60)
print('  Qwen2.5-VL (7B) \u2014 Vision-Language Model')
print('=' * 60)
qw_name, qw_m, qw_report = run_classifiers(qwen_feat[tr_idx], y_tr, qwen_feat[ev_idx], y_ev, 'Qwen')
print(f'\n  Best: {qw_name}')
print(qw_report)

## Step 5: Final Comparison

In [11]:
print('=' * 62)
print('  FINAL COMPARISON — HMDB51 (51 classes, ~6754 videos)')
print('=' * 62)

print(f'\n  {"Metric":<25} {"V-JEPA 2 (654M)":>18} {"Qwen2.5-VL (7B)":>18}')
print(f'  {"-"*61}')

def row(name, v1, v2, pct=True):
    if pct: print(f'  {name:<25} {v1*100:>17.1f}% {v2*100:>17.1f}%')
    else:   print(f'  {name:<25} {v1:>18.3f} {v2:>18.3f}')

row('Accuracy (top-1)',   vj_m['accuracy'],          qw_m['accuracy'])
if 'top5_accuracy' in vj_m and 'top5_accuracy' in qw_m:
    row('Accuracy (top-5)',   vj_m['top5_accuracy'],     qw_m['top5_accuracy'])
row('Precision (macro)',  vj_m['precision_macro'],   qw_m['precision_macro'])
row('Precision (weighted)', vj_m['precision_weighted'], qw_m['precision_weighted'])
row('Recall (macro)',     vj_m['recall_macro'],      qw_m['recall_macro'])
row('Recall (weighted)',  vj_m['recall_weighted'],   qw_m['recall_weighted'])
row('F1 (macro)',         vj_m['f1_macro'],          qw_m['f1_macro'])
row('F1 (weighted)',      vj_m['f1_weighted'],       qw_m['f1_weighted'])
row('MCC',                vj_m['mcc'],               qw_m['mcc'], pct=False)
row("Cohen's Kappa",      vj_m['cohen_kappa'],       qw_m['cohen_kappa'], pct=False)

print(f'\n  {"-"*61}')
print(f'  {"Architecture":<25} {"V-JEPA 2":>18} {"Qwen2.5-VL":>18}')
print(f'  {"-"*61}')
print(f'  {"Total params":<25} {"654M":>18} {"7B":>18}')
print(f'  {"Feature dim":<25} {str(vjepa_feat.shape[1]):>18} {str(qwen_feat.shape[1]):>18}')
print(f'  {"Best classifier":<25} {vj_name:>18} {qw_name:>18}')
print(f'  {"Paradigm":<25} {"Self-supervised":>18} {"Autoregressive":>18}')
print(f'  {"Modality":<25} {"Video only":>18} {"Vision+Language":>18}')

vj_f1 = vj_m['f1_macro']
qw_f1 = qw_m['f1_macro']
winner = 'V-JEPA 2' if vj_f1 > qw_f1 else 'Qwen2.5-VL'
margin = abs(vj_f1 - qw_f1) * 100

print(f'\n{"="*62}')
print(f'  WINNER: {winner} (by {margin:.1f}% F1)')
print(f'{"="*62}')

# Save results
results = {
    'dataset': 'HMDB51', 'num_classes': NUM_CLASSES,
    'num_videos': n, 'train': len(tr_idx), 'eval': len(ev_idx),
    'vjepa2': {'params': '654M', 'best_clf': vj_name,
               **{k: float(v) for k, v in vj_m.items()}},
    'qwen25vl': {'params': '7B', 'best_clf': qw_name,
                 **{k: float(v) for k, v in qw_m.items()}},
    'winner': winner,
}
results_path = os.path.join(SAVE_DIR, 'hmdb51_comparison.json')
json.dump(results, open(results_path, 'w'), indent=2)
print(f'\nResults saved: {results_path}')

  FINAL COMPARISON — HMDB51 (51 classes, ~6754 videos)

  Metric                       V-JEPA 2 (654M)    Qwen2.5-VL (7B)
  -------------------------------------------------------------
  Accuracy (top-1)                       80.8%              74.3%
  Accuracy (top-5)                       96.2%              93.9%
  Precision (macro)                      82.7%              78.5%
  Precision (weighted)                   81.7%              75.8%
  Recall (macro)                         81.3%              76.3%
  Recall (weighted)                      80.8%              74.3%
  F1 (macro)                             81.5%              76.8%
  F1 (weighted)                          80.8%              74.5%
  MCC                                    0.802              0.736
  Cohen's Kappa                          0.802              0.736

  -------------------------------------------------------------
  Architecture                        V-JEPA 2         Qwen2.5-VL
  ---------------------